In [ ]:
import os
import PIL.Image
import pytesseract
import pdf2image
from typing import Literal, Optional
import re
import pandas as pd
import cv2
import numpy as np

## Convert pdf to images

In [ ]:
def pdf_to_img(pdf_file, dpi: int=300):
    return pdf2image.convert_from_path(pdf_file, dpi=300)

def save_images(imgs, output_base="./output", file_format: Literal["PNG", "JPG"] = "PNG"):
    os.makedirs(output_base, exist_ok=True)
    for i, img in enumerate(imgs):
        img.save(os.path.join(output_base, f"page_{i}.{file_format}"), file_format)

In [ ]:
pdfFile = "input/MinnaNoNihongoSolutionsPartB.pdf"
imgs = pdf_to_img(pdfFile)

## Perform OCR with Tesseract

In [ ]:
def ocr_core(img, language: str = "jpn", custom_config: Optional[str] = None):
    if custom_config is None:
        custom_config = rf'--oem 3 --psm 6 -l {language}'
    text = pytesseract.image_to_string(img, lang=language)
    return text

def output_text(imgs, custom_config: Optional[str] = None):
    if custom_config is None:
        custom_config = r'--oem 3 --psm 6 -l jpn'
    for i, img in enumerate(imgs):
        text = pytesseract.image_to_string(img, config=custom_config)
        print(f"Page {i+1} Extracted Text:\n", text)

In [ ]:
text = ocr_core(imgs[1])
text2 = ocr_core(imgs[2])
print(text)

In [ ]:
# save_images(imgs)
# output_text(imgs)

In [ ]:
print(text)

In [ ]:
print(text2)

## Extract Structured Data
Since the document has a clear structure (第X課, 練習B, X., 1) 2) ...), we can use regular expressions to extract information.

**Regex patterns**:
- **Chapters**: `第(\d+)課`
- **Section B**: `練習B`
- **Exercises**: `(\d+)\.` (number followed by a dot)
- **Sentences**: `(\d+)\)` (number followed by a close round bracket)

In [ ]:
def extract_structured_text(text):
    structured_data = {}
    current_chapter = None
    current_exercise = None

    lines = text.split("\n")
    for line in lines:
        print(line)
        a = re.match("^練習 B$", line)
        print(a)
        # Detect chapter
        chapter_match = re.search(r'第(\d+)課', line)
        if chapter_match:
            current_chapter = int(chapter_match.group(1))
            structured_data[current_chapter] = {}

        # Do not continue until a chapter has been identified
        if current_chapter is None:
            continue

        # Detect "練習B"
        if "練習 B" in line:
            print("Found Renshuu B")
            structured_data[current_chapter]["練習 B"] = {}

        # Detect exercises (number followed by a dot)
        exercise_match = re.match(r'^(\d+)\.', line.strip())
        if exercise_match:
            current_exercise = int(exercise_match.group(1))
            structured_data[current_chapter]["練習 B"][current_exercise] = []

        # Detect sentences (number followed by `)`)
        sentence_matches = re.findall(r'(\d+)\)', line)
        if sentence_matches and current_exercise:
            sentences = re.split(r'\d+\)', line)[1:]  # Remove numbers
            sentences = [s.strip() for s in sentences if s.strip()]
            structured_data[current_chapter]["練習 B"][current_exercise].extend(sentences)

    return structured_data

In [ ]:
struc = extract_structured_text(text)

## Store in Excel for Easy Access
Once we have structured data, we save it in an Excel file for reloading later.

In [ ]:
def save_to_excel(data, filename="extracted_text.xlsx"):
    rows = []

    for chapter, sections in data.items():
        for section, exercises in sections.items():
            for exercise, sentences in exercises.items():
                for sentence_num, sentence in enumerate(sentences, 1):
                    rows.append([chapter, section, exercise, sentence_num, sentence])

    df = pd.DataFrame(rows, columns=["Chapter", "Section", "Exercise", "Sentence Number", "Sentence"])
    df.to_excel(filename, index=False)

### Reload Data for Easy Access

In [ ]:
def get_sentence(chapter, exercise, sentence_number, filename="extracted_text.xlsx"):
    df = pd.read_excel(filename)
    result = df[
        (df["Chapter"] == chapter) &
        (df["Exercise"] == exercise) &
        (df["Sentence Number"] == sentence_number)
    ]
    return result["Sentence"].values[0] if not result.empty else None

In [ ]:
# Example usage
sentence = get_sentence(3, 5, 2)  # Get sentence 2 of exercise 5 in chapter 3
print(sentence)

## Optional: Improve OCR Accuracy
If OCR results are messy, you can:

- Preprocess images (binarization, noise removal) before passing to Tesseract.
- Use a better OCR engine like Google Vision API for improved accuracy.

**Example of binarization:**

In [ ]:
def preprocess_image(pil_img):
    # Convert PIL image to grayscale first
    pil_gray = pil_img.convert("L")  # "L" mode means grayscale

    # Convert grayscale PIL image to NumPy array
    np_gray = np.array(pil_gray)

    # Ensure it's a contiguous array
    np_gray = np.ascontiguousarray(np_gray)

    # Apply binary thresholding (Otsu's method)
    _, binary = cv2.threshold(np_gray, 150, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    return binary

def binarized_ocr(img):
    bin_img = preprocess_image(img)
    text = ocr_core(bin_img)
    return text